По базе машин с ЮЛЫ данным обучите модель для предсказания цен на машины.

1. Создайте обучающую, тестовую и проверочную выборки.

2. Оцените качество работы созданной сети, определив средний процент ошибки на проверочной выборке. (Для этого потребуется привести предсказанные моделью значения к первоначальному диапазону цен.)  

3. Подсчитайте ошибку на каждом примере тестовой выборки и суммарный процент ошибки.


Рекомендации:
- в качестве ошибки рекомендуется использовать среднеквадратическую ошибку (mse).
- метрику для данной задачи можно не использовать.
- последний слой модели должен иметь 1 нейрон.
- суммарный процент ошибки = средний модуль ошибки (MAE) / среднюю цену машины. Например, если средняя цена машины 560.000 р, а средняя ошибка 56.000р, то процент ошибки равен 10%.


In [16]:
# Загрузка данных
import gdown
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error

gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l10/cars_new.csv', None, quiet=True)

df = pd.read_csv('cars_new.csv')
print(f"Размер данных: {df.shape}")

Размер данных: (70119, 10)


In [18]:
# Обработка категориальных признаков
X = pd.get_dummies(df.drop('price', axis=1), drop_first=True)
y = df['price'].values.astype('float32')

print(f"Признаки после кодирования: {X.shape}")
print(f"Цены: {y.shape}")

Признаки после кодирования: (70119, 3201)
Цены: (70119,)


In [19]:
# Масштабирование
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

In [20]:
# Разделение на обучающую, проверочную и тестовую выборки
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_scaled, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Обучающая: {X_train.shape[0]}")
print(f"Проверочная: {X_val.shape[0]}")
print(f"Тестовая: {X_test.shape[0]}")

Обучающая: 49083
Проверочная: 10518
Тестовая: 10518


In [ ]:
# Модель

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential()
model.add(Dense(128, activation='relu', input_dim=X_train.shape[1]))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='linear'))

model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                │ (None, 128)            │       409,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 420,993 (1.61 MB)

 Trainable params: 420,609 (1.60 MB)

 Non-trainable params: 384 (1.50 KB)

In [23]:
# Обучение

from tensorflow.keras.callbacks import EarlyStopping, LambdaCallback

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)

def print_mae(epoch, logs):
    pred_scaled = model.predict(X_val, verbose=0)
    pred_real = scaler_y.inverse_transform(pred_scaled).flatten()
    y_val_real = scaler_y.inverse_transform(y_val.reshape(-1, 1)).flatten()
    mae_rub = np.mean(np.abs(pred_real - y_val_real))
    print(f"  → MAE на проверочной: {mae_rub:.0f} руб")

callback = LambdaCallback(on_epoch_end=print_mae)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=32, verbose=1,
    callbacks=[early_stop, callback]
)

Epoch 1/50
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.8389 - mae: 0.5218  → MAE на проверочной: 139188 руб
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - loss: 0.6503 - mae: 0.4181 - val_loss: 0.4389 - val_mae: 0.2232
Epoch 2/50
1518/1534 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4958 - mae: 0.3057  → MAE на проверочной: 133830 руб
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.3689 - mae: 0.2886 - val_loss: 0.4120 - val_mae: 0.2146
Epoch 3/50
1517/1534 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2714 - mae: 0.2540  → MAE на проверочной: 116350 руб
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.2931 - mae: 0.2524 - val_loss: 0.4142 - val_mae: 0.1866
Epoch 4/50
1517/1534 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2203 - mae: 0.2284  → MAE на проверочной: 118597 руб
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.2488 - mae: 0.2293 - val_loss: 0.4221 - val_mae: 0.1902
Epoch 5/50
1523/1534 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1983 - mae: 0.2161  → MAE на пр

In [24]:
# Ошибка на каждом примере и суммарный процент ошибки

y_pred_scaled = model.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()
y_test_real = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

errors = np.abs(y_test_real - y_pred)

print("Ошибки на первых 10 примерах тестовой выборки:")
for i in range(10):
    print(f"  Пример {i+1}: реальная = {y_test_real[i]:.0f} руб, предсказано = {y_pred[i]:.0f} руб, ошибка = {errors[i]:.0f} руб")

mean_price = np.mean(y_test_real)
mean_error = np.mean(errors)
error_percent = (mean_error / mean_price) * 100

print(f"\nСредняя цена машины: {mean_price:.0f} руб")
print(f"Средняя ошибка (MAE): {mean_error:.0f} руб")
print(f"MSE: {mean_squared_error(y_test_real, y_pred):.2f}")
print(f"Суммарный процент ошибки: {error_percent:.2f}%")

329/329 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Ошибки на первых 10 примерах тестовой выборки:
  Пример 1: реальная = 370000 руб, предсказано = 431977 руб, ошибка = 61977 руб
  Пример 2: реальная = 100000 руб, предсказано = 151006 руб, ошибка = 51006 руб
  Пример 3: реальная = 295000 руб, предсказано = 325051 руб, ошибка = 30051 руб
  Пример 4: реальная = 140000 руб, предсказано = 213639 руб, ошибка = 73639 руб
  Пример 5: реальная = 1150000 руб, предсказано = 1095452 руб, ошибка = 54548 руб
  Пример 6: реальная = 1499000 руб, предсказано = 1291013 руб, ошибка = 207987 руб
  Пример 7: реальная = 165000 руб, предсказано = 170845 руб, ошибка = 5845 руб
  Пример 8: реальная = 190000 руб, предсказано = 191956 руб, ошибка = 1956 руб
  Пример 9: реальная = 540000 руб, предсказано = 433295 руб, ошибка = 106705 руб
  Пример 10: реальная = 335000 руб, предсказано = 310703 руб, ошибка = 24297 руб

Средняя цена машины: 523842 руб
Средняя ошибка (MAE): 88234 руб
MSE: 71342768128.00
Суммарный процент ошиб